# Práctica 4: Embeddings distribucionales y contextuals

## Introducción Teórica y Objetivos
El objetivo de la práctica es entrenar y evaluar modelos de **embeddings distribucionales (estáticos)** y compararlos con **embeddings contextuales** para tareas de similitud semántica.

Los **Word Embeddings** permiten representar las palabras como vectores numéricos basándose en su contexto.
1. **Estáticos (Word2Vec, fastText)**: Cada palabra tiene siempre el mismo vector. Compararemos el comportamiento frente a palabras fuera del vocabulario (OOV) y el efecto del tamaño del vector y del corpus de entrenamiento.
2. **Contextuales (BERT, RoBERTa)**: Generan representaciones dinámicas basadas en la oración entera, lo que resuelve problemas como la polisemia.



## Evaluación
* **Intrínseca (Multi-SimLex)**: Evaluaremos la distancia coseno entre vectores asociados para obtener la similitud y calcularemos la **correlación de Spearman** comparada con evaluadores humanos.
* **Extrínseca (Spanish STS)**: Calcularemos la similitud semántica de documentos enteros (frases) y utilizaremos la **correlación de Pearson**.



## Datasets
- **Multi-SimLex (Spanish):** conjunto de pares de palabras con puntuaciones de similitud semántica anotadas por humanos. Se utiliza para la **evaluación intrínseca**, midiendo directamente la calidad de los embeddings mediante la **correlación de Spearman** entre las similitudes predichas y las humanas.

- **Spanish STS:** conjunto de pares de frases con puntuaciones de similitud semántica. Se utiliza para la **evaluación extrínseca**, midiendo la **correlación de Pearson** entre las similitudes predichas y las de referencia.

- **Corpus Wikipedia en español (`raw.es.tgz`):** corpus de texto en crudo utilizado para entrenar los embeddings distribucionales.

## 0. Setup

In [ ]:
# Instalar dependencias si es necesario
# %pip install pandas scipy datasets tqdm gensim torch sentence-transformers scikit-learn transformers
# pip install -U datasets

import pandas as pd
import numpy as np
import os
import re
from scipy.stats import spearmanr, pearsonr
from datasets import load_dataset
from gensim.models import Word2Vec, FastText
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

# %pip install requests "datasets<3.0"
# from pathlib import Path
# from tqdm.notebook import tqdm
# import nltk
# from itertools import product
# from sklearn.model_selection import KFold

ModuleNotFoundError: No module named 'nltk'

## 1. Datasets

Esta práctica utiliza tres datasets:

| Dataset | Descripción | Uso |
|---|---|---|
| **Multi-SimLex** | Pares de palabras con puntuaciones de similitud semántica | Evaluación intrínseca |
| **Spanish STS** | Pares de frases con puntuaciones de similitud semántica | Evaluación extrínseca |
| **raw_es** | Corpus de Wikipedia en español | Entrenamiento de embeddings |

A partir de `raw_es`, entrenaremos y compararemos modelos de **Word2Vec** y **fastText**.

## Columnas del dataset Multi-SimLex (ES)

| Columna | Descripción |
|---|---|
| **PAIR_ID** | Identificador único del par de palabras en el dataset. |
| **SPA_W1** | Primera palabra del par. |
| **SPA_W2** | Segunda palabra del par. |
| **SPA_AVG** | Puntuación media de similitud semántica asignada por varios anotadores humanos. Cuanto mayor es el valor, más similares se consideran las dos palabras. En este dataset los valores observados van aproximadamente de **0.0** a **5.9**. |

In [2]:
# ============================================================
# DATASET 1 — Multi-SimLex
# ============================================================

os.system("wget -O SPA.csv https://web.archive.org/web/20231020014354/https://multisimlex.com/data/SPA.csv")

spa = pd.read_csv("SPA.csv")

annotator_cols = [c for c in spa.columns if c.startswith('Annotator')]
spa['SPA_AVG'] = spa[annotator_cols].mean(axis=1)

multi_simlex = spa[['ID', 'Word 1', 'Word 2', 'SPA_AVG']].copy()
multi_simlex.columns = ['PAIR_ID', 'SPA_W1', 'SPA_W2', 'SPA_AVG']

print(f"Multi-SimLex (ES) — {len(multi_simlex)} pares de palabras")
multi_simlex.head()

Multi-SimLex (ES) — 1888 pares de palabras


,PAIR_ID,SPA_W1,SPA_W2,SPA_AVG
0,1,brazo,músculo,1.4
1,2,democracia,monarquía,1.3
2,3,tejado,techo,4.8
3,4,amigo,profesor,0.4
4,5,mano,pie,1.1


In [3]:
# ver valores mínimo y máximo de las puntuaciones
min_score = multi_simlex['SPA_AVG'].min()
max_score = multi_simlex['SPA_AVG'].max()

print("Valor mínimo:", min_score)
print("Valor máximo:", max_score)

Valor mínimo: 0.0
Valor máximo: 5.9


## Columnas del dataset Spanish STS

| Columna | Descripción |
|---|---|
| **id** | Identificador único del par de frases. |
| **sentence1** | Primera frase del par. |
| **sentence2** | Segunda frase del par. |
| **score** | Puntuación de similitud semántica asignada por anotadores humanos. Cuanto mayor es el valor, más similares son las dos frases. |

In [ ]:
# ============================================================
# DATASET 2 — Spanish STS (PlanTL-GOB-ES/sts-es)
# Avaluación extrínseca · Mètrica: correlación de Pearson
# ============================================================
sts = load_dataset("PlanTL-GOB-ES/sts-es")

train_df = sts["train"].to_pandas().rename(columns={"label": "score"})
dev_df   = sts["validation"].to_pandas().rename(columns={"label": "score"})
test_df  = sts["test"].to_pandas().rename(columns={"label": "score"})

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")
print(train_df.head())

Generating test split: 100%|██████████| 155/155 [00:00<?, ? examples/s]

Train: 1320 | Dev: 77 | Test: 155
  id                                          sentence1  \
0  0  Según el sondeo, 87% de los católicos cree que...   
1  1  La psicología explora conceptos como la percep...   
2  2  La tradición comenzó en el siglo IV, pero la m...   
3  3  "Maria Anna Schwegelin" (? - 1781 en la cárcel...   
4  4  Su identidad la había revelado durante el viaj...   

                                           sentence2  score  
0  El 87% de los católicos del mundo aprobaron el...   3.75  
1  La "psicología básica" es la parte de la psico...   2.80  
2  La tradición de erigir piedras con inscripcion...   2.40  
3  Te entregamos, Anna Schwegelin, al verdugo par...   2.20  
4  La información fue suministrada por el pescado...   2.20  


## Corpus de Wikipedia en español (`raw_es`)

Este corpus se utilizará para entrenar los modelos de embeddings (**Word2Vec** y **fastText**).

El corpus está dividido en múltiples archivos de texto dentro de la carpeta `raw_es`.

Cada archivo contiene una parte del texto extraído de Wikipedia en español.

En total se han encontrado **57 archivos** dentro del corpus.

In [ ]:
# ============================================================
# DATASET 3 — Corpus Wikipedia en español
# ============================================================
corpus_dir   = "raw_es"
corpus_files = sorted(os.listdir(corpus_dir))

print(f"Total archivos encontrados: {len(corpus_files)}")
print(corpus_files[:5])

Total fitxers trobats: 57
['spanishText_10000_15000', 'spanishText_110000_115000', 'spanishText_120000_125000', 'spanishText_15000_20000', 'spanishText_180000_185000']


## 2. Preprocessament de *raw_es*

Primero, es necesario realizar un preprocesamiento en los corpus de *raw_es* para eliminar elementos que no nos sirven: **</doc ...>**, **ENDOFARTICLE** y **URLs**. 
También realizaremos otros preprocesamientos más básicos: eliminar *stopwords*, convertir a minúsculas, eliminar espacios múltiples y tokenizar.

In [11]:
#%pip install nltk
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\crost\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
import unicodedata

STOPWORDS = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    # Normalizar Unicode (muy importante en dumps Wikipedia)
    texto = unicodedata.normalize("NFKD", texto)

    # Minúsculas
    texto = texto.lower()

    # Eliminar URLs
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)

    # Eliminar etiquetas XML tipo <doc ...> o </doc>
    texto = re.sub(r'<[^>]+>', ' ', texto)

    # Eliminar ENDOFARTICLE
    texto = re.sub(r'endofarticle\.?', ' ', texto, flags=re.IGNORECASE)

    # Mantener solo letras y espacios
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)

    # Normalizar espacios
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto


def preprocessa_linea(linea):
    linea = linea.strip()
    if not linea:
        return []

    texto = limpiar_texto(linea)

    palabras = [
        p for p in texto.split()
        if p and p not in STOPWORDS
    ]

    return palabras


def cargar_corpus(directorio, archivos):
    for nombre_archivo in archivos:
        ruta = os.path.join(directorio, nombre_archivo)
        with open(ruta, "r", encoding="latin-1", errors="ignore") as f:
            for linea in f:
                palabras = preprocessa_linea(linea)
                if palabras:
                    yield palabras


corpus = cargar_corpus(corpus_dir, corpus_files)

print("Primeras 3 frases preprocesadas:")
for i, frase in enumerate(corpus):
    print(f"  [{i+1}] {frase[:10]}")
    if i == 2:
        break

Primeras 3 frases preprocesadas:
  [1] ['acontecimientos', 'nacimientos', 'fallecimientos', 'fulgencio', 'cija', 'santo', 'espan', 'ol', 'erquinoaldo', 'mayordomo']
  [2] ['acontecimientos', 'nacimientos', 'egilona', 'u', 'ltima', 'reina', 'visigoda', 'hispania', 'fallecimientos']
  [3] ['acontecimientos', 'fin', 'califato', 'perfecto', 'omeyas', 'poder', 'califato', 'damasco', 'divisio', 'n']


## 3. Models Word2Vec amb raw_es

El model Word2Vec és auto-supervisat, per tant no té un conjunt de validació amb etiquetes. El GridSearch requereix aquestes etiquetes per a la seva funció .fit(x.y), per tant no se li podrà aplicar. Com a solució,

In [9]:
class WikiCorpus:
    def __init__(self, directori, fitxers):
        self.directori = directori
        self.fitxers   = fitxers

    def __iter__(self):
        for nom_fitxer in self.fitxers:
            ruta = os.path.join(self.directori, nom_fitxer)
            with open(ruta, "r", encoding="latin-1") as f:
                for linia in f:
                    paraules = preprocessa_linia(linia)
                    if paraules:
                        yield paraules

### Word2Vec: CBOW

L'arquitectura *Continuous Bag of Words* de Word2Vec agafa el context i prediu la paraula central. En aquesta arquitectura, l'ordre de les paraules no importa, només quines paraules hi ha. Un exemple clàssic és quan volem completar una frase que li falta una paraula segons el context que ens dóna les altres paraules:
 - "El gat seu sobre la ___"

In [10]:
def avalua_model(model, parells_df, n_splits=5):
    """
    Calcula la correlació de Spearman mitjançant K-Fold Cross Validation
    sobre els parells de Multi-SimLex.
    Retorna la mitjana i desviació estàndard de les correlacions.
    """
    # Filtrar parells on ambdues paraules estan al vocabulari
    mascara = parells_df.apply(
        lambda r: r['SPA_W1'] in model.wv and r['SPA_W2'] in model.wv,
        axis=1
    )
    df_valid = parells_df[mascara].reset_index(drop=True)

    if len(df_valid) < n_splits:
        return np.nan, np.nan

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    correlacions = []

    for _, idx_test in kf.split(df_valid):
        fold = df_valid.iloc[idx_test]

        similituds_model = [
            model.wv.similarity(r['SPA_W1'], r['SPA_W2'])
            for _, r in fold.iterrows()
        ]
        similituds_gold = fold['SPA_AVG'].tolist()

        corr, _ = spearmanr(similituds_model, similituds_gold)
        correlacions.append(corr)

    return np.mean(correlacions), np.std(correlacions)

In [ ]:
    # ============================================================
    # GRID SEARCH RÀPID — Word2Vec CBOW
    # Estratègia: cercar hiperparàmetres sobre una mostra del corpus
    #             i ampliar el millor model amb més dades
    # ============================================================

# Si el model ja existeix, carreguem directament sense reentrenar
if os.path.exists("w2v_cbow_optimitzat.model"):
    print("Model ja existent, carregant w2v_cbow_optimitzat.model...")
    w2v_cbow = Word2Vec.load("w2v_cbow_optimitzat.model")
    print(f"Model carregat ✓ — vocabulari: {len(w2v_cbow.wv):,} paraules")

else:
    print("Model no trobat, iniciant entrenament...")´
    
    # ── FASE 1: CARREGAR MOSTRA A RAM ────────────────────────────────────────────
    N_FITXERS_MOSTRA = 5

    print(f"Carregant mostra ({N_FITXERS_MOSTRA} fitxers) a memòria...")
    corpus_mostra = list(WikiCorpus(corpus_dir, corpus_files[:N_FITXERS_MOSTRA]))
    print(f"Frases a la mostra: {len(corpus_mostra):,}")

    # ── FASE 2: GRID SEARCH SOBRE LA MOSTRA ─────────────────────────────────────
    param_grid = {
        'vector_size': [100, 300],
        'window':      [3, 5],
        'min_count':   [5, 10],
        'epochs':      [5, 10],
    }

    claus        = list(param_grid.keys())
    valors       = list(param_grid.values())
    combinacions = list(product(*valors))

    print(f"\nTotal combinacions: {len(combinacions)}\n")

    resultats    = []
    millor_model = None       # ← guardem el millor model durant el bucle
    millor_score = -np.inf

    for i, combo in enumerate(combinacions):
        params = dict(zip(claus, combo))
        print(f"[{i+1}/{len(combinacions)}] Entrenant amb {params}...")

        model = Word2Vec(
            sentences=corpus_mostra,
            vector_size=params['vector_size'],
            window=params['window'],
            min_count=params['min_count'],
            sg=0,
            workers=4,
            epochs=params['epochs']
        )

        mitjana, std = avalua_model(model, multi_simlex, n_splits=5)
        resultats.append({**params, 'spearman_mean': mitjana, 'spearman_std': std})
        print(f"  → Spearman: {mitjana:.4f} ± {std:.4f}")

        # Guardem el model si és el millor fins ara
        if mitjana > millor_score:
            millor_score = mitjana
            millor_model = model

    # ── RESULTATS ────────────────────────────────────────────────────────────────
    resultats_df = pd.DataFrame(resultats).sort_values('spearman_mean', ascending=False)
    print("\nRànquing de models:")
    print(resultats_df.to_string(index=False))

    millors = resultats_df.iloc[0]
    print(f"\nMillors hiperparàmetres:")
    print(f"  vector_size : {int(millors['vector_size'])}")
    print(f"  window      : {int(millors['window'])}")
    print(f"  min_count   : {int(millors['min_count'])}")
    print(f"  epochs      : {int(millors['epochs'])}")
    print(f"  Spearman    : {millors['spearman_mean']:.4f} ± {millors['spearman_std']:.4f}")

    # ── FASE 3: AMPLIAR EL MILLOR MODEL AMB MÉS DADES ───────────────────────────
    N_FITXERS_FINAL = 20

    print(f"\nAmpliant el millor model amb {N_FITXERS_FINAL} fitxers...")
    corpus_final = list(WikiCorpus(corpus_dir, corpus_files[:N_FITXERS_FINAL]))

    # Ampliar el vocabulari amb les noves dades
    millor_model.build_vocab(corpus_final, update=True)

    # Continuar l'entrenament sobre les noves dades
    millor_model.train(
        corpus_final,
        total_examples=len(corpus_final),
        epochs=millor_model.epochs
    )

    w2v_cbow = millor_model
    w2v_cbow.save("w2v_cbow_optimitzat.model")
    print("Model CBOW optimitzat guardat ✓")

    # tarda 10 min casi

Carregant mostra (5 fitxers) a memòria...
Frases a la mostra: 531,869

Total combinacions: 16

[1/16] Entrenant amb {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 5}...
  → Spearman: 0.2619 ± 0.0278
[2/16] Entrenant amb {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 10}...
  → Spearman: 0.3087 ± 0.0378
[3/16] Entrenant amb {'vector_size': 100, 'window': 3, 'min_count': 10, 'epochs': 5}...
  → Spearman: 0.2631 ± 0.0267
[4/16] Entrenant amb {'vector_size': 100, 'window': 3, 'min_count': 10, 'epochs': 10}...
  → Spearman: 0.3119 ± 0.0270
[5/16] Entrenant amb {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 5}...
  → Spearman: 0.2755 ± 0.0374
[6/16] Entrenant amb {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 10}...
  → Spearman: 0.3242 ± 0.0436
[7/16] Entrenant amb {'vector_size': 100, 'window': 5, 'min_count': 10, 'epochs': 5}...
  → Spearman: 0.2851 ± 0.0284
[8/16] Entrenant amb {'vector_size': 100, 'window': 5, 'min_count': 10, 'epochs'

Importem el model òptim guardat.

In [14]:
from gensim.models import Word2Vec

w2v_cbow = Word2Vec.load("w2v_cbow_optimitzat.model")

print(f"Vocabulari: {len(w2v_cbow.wv):,} paraules")
print(f"Dimensió dels vectors: {w2v_cbow.vector_size}")

Vocabulari: 172,689 paraules
Dimensió dels vectors: 100


### Word2Vec: Skip-gram

L'estructura *Skip-gram* de Word2Vec fa el procés invers a CBOW: ara sabem la paraula central, així que volem predir les paraules que estiguin al voltant (el context). Aquest model prediu per a cada posició de la finestra de manera independent, per tant pot generar molts exemples possibles, i això fa que el seu entrenament sigui més lent que CBOW. L'avantatge és que aprèn millor les paraules menys freqüents: com que genera molts exemples per cada paraula, fins i tot les poc freqüents es veuen prou vegades per aprendre una bona representació.

## 4. Models FastText